# 05 – Next-Gen Pipeline: RT-DETR + TrOCR
**ICS472 – Natural Language Processing**  
**Team:** Mohammed Al Sheqaih · Abdulrhman Ammar

## Architecture Overview

| Part | Model | Key Improvement |
|---|---|---|
| A | **RT-DETR-L** | Transformer detector; GIoU loss; rect inference; GPU perspective aug |
| B | **TrOCR** | ViT encoder + Transformer decoder for digit sequences |
| C | **TrOCR + Arabic preproc** | H-flip (RTL→LTR); 4× augmentation; char tokeniser (24 labels); spelling normalisation |
| D | **Levenshtein + Magnitude** | Fuzzy monetary-vocab matching; missing-zero magnitude correction |

**Hardware target:** NVIDIA RTX 5060 — AMP + gradient checkpointing throughout.

**Pipeline:**
```
Check image
    └─ RT-DETR ───────┬─ Courtesy crop ── TrOCR-B ── parse_int ─┬───┬─ Levenshtein+
                    └─ Legal crop ──── TrOCR-C ── parse_int ─┘   └─ Magnitude ─── Verdict
```

> **Prerequisite:** Run `02_Part_B_Courtesy.ipynb` and `03_Part_C_Legal_Improved.ipynb`
> to produce `artifacts/partD_metrics.json` for the comparison plot in Part D.

## 0. Install Dependencies
Run the cell below once per environment (comment out afterwards).

In [5]:
import subprocess, sys

pkgs = [
    'ultralytics>=8.1',       # RT-DETR support
    'transformers>=4.36',     # TrOCR
    'datasets',
    'accelerate',
    'evaluate',
    'jiwer',                  # CER / WER
    'albumentations',         # image augmentation
    'python-Levenshtein',     # fast edit distance
    'pyyaml',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('All packages installed.')

All packages installed.


## 1. Imports & Setup

In [1]:
import sys, os, json, re, shutil
from pathlib import Path
from typing import List, Optional, Tuple
from collections import Counter

import numpy as np
import torch
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath('.'))
from utils import (
    ARTIFACTS, ROOT,
    TRAIN_IMAGES, TEST_IMAGES,
    TRAIN_BBOX,   TEST_BBOX,
    TRAIN_CA, TRAIN_LA,
    TEST_CA,  TEST_LA,
    parse_bbox, legal_text_to_digits,
)

plt.rcParams['figure.dpi'] = 120

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}')
    print(f'VRAM   : {props.total_memory / 1e9:.1f} GB')

NEXTGEN = ARTIFACTS / 'nextgen'
NEXTGEN.mkdir(parents=True, exist_ok=True)
print(f'Output : {NEXTGEN}')

Device : cuda
GPU    : NVIDIA GeForce RTX 5060
VRAM   : 8.5 GB
Output : C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\nextgen


---
## Part A — Detection with RT-DETR

RT-DETR (Real-Time DEtection TRansformer) replaces YOLO with a Transformer decoder.
Key improvements for bank-check detection:

| Feature | Benefit |
|---|---|
| **GIoU / L1 loss** | Tighter bounding boxes at IoU ≥ 0.75 — cleaner crops fed to OCR |
| **Multi-scale feature maps** | Handles the wide aspect ratio of courtesy/legal regions |
| **`rect=True` inference** | No aspect-ratio distortion for wide check images |
| **GPU Perspective aug** | Synthesises tilted-check variations without CPU bottleneck |
| **Mosaic aug** | Increases effective training set size |

### A1. Build Dataset YAML

In [2]:
import yaml

# RT-DETR (via ultralytics) accepts the same YOLO-format labels already in the workspace.
# class 0 = legal amount region  |  class 1 = courtesy amount region

DATA_DIR = NEXTGEN / 'rtdetr_data'
for split in ('train', 'val'):
    img_split_dir = DATA_DIR / 'images' / split
    img_split_dir.mkdir(parents=True, exist_ok=True)
    # Wipe ALL files (any leftover .tif / bad .png from previous runs)
    for f in img_split_dir.iterdir():
        f.unlink()
    (DATA_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print('Image directories cleaned.')


def collect_stems(img_dir: Path, bbox_dir: Path) -> List[str]:
    stems = []
    for img in sorted(img_dir.glob('*.tif')):
        if (bbox_dir / (img.stem + '.txt')).exists():
            stems.append(img.stem)
    return stems


def populate_split(stems, img_dir, bbox_dir, split):
    """Save every image as a proper 3-channel RGB PNG."""
    img_out = DATA_DIR / 'images' / split
    lbl_out = DATA_DIR / 'labels' / split
    for stem in stems:
        img = Image.open(img_dir / f'{stem}.tif').convert('RGB')
        img.save(img_out / f'{stem}.png')
        dst_lbl = lbl_out / f'{stem}.txt'
        if not dst_lbl.exists():
            shutil.copy2(bbox_dir / f'{stem}.txt', dst_lbl)


train_stems = collect_stems(TRAIN_IMAGES, TRAIN_BBOX)
test_stems  = collect_stems(TEST_IMAGES,  TEST_BBOX)
print(f'Train stems : {len(train_stems)}')
print(f'Test  stems : {len(test_stems)}')

print('Converting train images to RGB PNG...')
populate_split(train_stems, TRAIN_IMAGES, TRAIN_BBOX, 'train')
print('Converting val images to RGB PNG...')
populate_split(test_stems,  TEST_IMAGES,  TEST_BBOX,  'val')

# Verify — every file in the image dirs must be 3-channel
print('Verifying channel counts...')
bad = []
for split in ('train', 'val'):
    for p in (DATA_DIR / 'images' / split).iterdir():
        arr = np.array(Image.open(p))
        if arr.ndim != 3 or arr.shape[2] != 3:
            bad.append(str(p))
if bad:
    raise RuntimeError(f'Found {len(bad)} non-RGB files:\n' + '\n'.join(bad))
print('All images verified as RGB.')

data_yaml = {
    'path':  str(DATA_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    2,
    'names': {0: 'legal', 1: 'courtesy'},
}
yaml_path = DATA_DIR / 'check_detect.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)
print(f'Dataset YAML written to {yaml_path}')


Image directories cleaned.
Train stems : 1799
Test  stems : 600
Converting train images to RGB PNG...
Converting val images to RGB PNG...
Verifying channel counts...
All images verified as RGB.
Dataset YAML written to C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\nextgen\rtdetr_data\check_detect.yaml


### A2. Train RT-DETR-L

RTX 5060 config: `batch=8`, `amp=True` (Tensor Cores), `rect=True` (wide checks),
`perspective=0.001` + `mosaic=1.0` (augmentation).

In [3]:
from ultralytics import RTDETR

model_a = RTDETR('rtdetr-l.pt')   # Large variant ~32 M params


def on_epoch_end(trainer):
    e      = trainer.epoch + 1
    total  = trainer.epochs
    loss   = trainer.loss.item() if hasattr(trainer.loss, 'item') else float(trainer.loss)
    metrics = trainer.metrics or {}
    map50  = metrics.get('metrics/mAP50(B)', float('nan'))
    map50_95 = metrics.get('metrics/mAP50-95(B)', float('nan'))
    lr     = trainer.optimizer.param_groups[0]['lr']
    print(f'  Epoch {e:>3}/{total}  |  loss: {loss:.4f}  |  '
          f'mAP@50: {map50:.4f}  |  mAP@50-95: {map50_95:.4f}  |  lr: {lr:.2e}')


model_a.add_callback('on_train_epoch_end', on_epoch_end)

# Fix for mosaic channel-mismatch error:
# 1) force all images to RGB again
# 2) remove stale Ultralytics cache files
# 3) disable mosaic (error originates inside _mosaic4)

for split in ('train', 'val'):
    for p in (DATA_DIR / 'images' / split).glob('*.png'):
        Image.open(p).convert('RGB').save(p)

for cache_file in DATA_DIR.rglob('*.cache'):
    cache_file.unlink()

results_a = model_a.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=8,
    rect=True,
    mosaic=0.0,          # disable to avoid _mosaic4 channel broadcast crash
    amp=True,
    device=0,
    workers=4,
    cache=False,         # force fresh reads after RGB rewrite
    patience=20,
    save_period=10,
    project=str(NEXTGEN / 'rtdetr_run'),
    name='detect',
    exist_ok=True,
    verbose=False,
)

BEST_A = Path(results_a.save_dir) / 'weights' / 'best.pt'
print(f'RT-DETR training complete. Best weights: {BEST_A}')


New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.10.19 torch-2.12.0.dev20260219+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\nextgen\rtdetr_data\check_detect.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_de

c:\Users\Admin\miniconda3\envs\warehouse_ai\lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      6.53G     0.9344     0.9528      0.756         14        640: 100% ━━━━━━━━━━━━ 225/225 2.3it/s 1:39<0.5ss
  Epoch   1/100  |  loss: 15.2029  |  mAP@50: 0.0000  |  mAP@50-95: 0.0000  |  lr: 5.53e-04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 4.7it/s 8.2s0.2s
                   all        600       1200      0.654      0.683      0.571      0.182

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      6.63G     0.4165     0.4184     0.2015         16        640: 0% ──────────── 0/225  0.6s

c:\Users\Admin\miniconda3\envs\warehouse_ai\lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      6.63G      0.538       0.41     0.2853         14        640: 100% ━━━━━━━━━━━━ 225/225 2.5it/s 1:31<0.4ss
  Epoch   2/100  |  loss: 13.2364  |  mAP@50: 0.5708  |  mAP@50-95: 0.1819  |  lr: 1.10e-03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 5.4it/s 7.0s0.2s
                   all        600       1200      0.739      0.732      0.676      0.203

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      6.73G     0.4474     0.3871     0.2029         16        640: 0% ──────────── 0/225  0.5s

...

     42/100      6.85G     0.3945     0.3676     0.1822         13        640: 100% ━━━━━━━━━━━━ 225/225 2.2it/s 1:42<0.5ss
  Epoch  42/100  |  loss: 10.4075  |  mAP@50: 0.6446  |  mAP@50-95: 0.2422  |  lr: 9.90e-04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 4.6it/s 8.2s0.2s
                   all        600       1200       0.64      0.652       0.56      0.189

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     43/100      6.72G       0.26     0.3353     0.1595         15        640: 0% ──────────── 0/225  0.5s

c:\Users\Admin\miniconda3\envs\warehouse_ai\lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     43/100      6.72G     0.3978     0.3696     0.1857         13        640: 100% ━━━━━━━━━━━━ 225/225 2.0it/s 1:50<0.5ss
  Epoch  43/100  |  loss: 11.6135  |  mAP@50: 0.5601  |  mAP@50-95: 0.1889  |  lr: 9.74e-04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 4.6it/s 8.2s0.2s
                   all        600       1200      0.689      0.693      0.627      0.254

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     44/100      6.72G     0.5155     0.3755     0.1854         16        640: 0% ──────────── 0/225  0.5s

c:\Users\Admin\miniconda3\envs\warehouse_ai\lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     44/100      6.72G     0.3947     0.3657     0.1847         13        640: 100% ━━━━━━━━━━━━ 225/225 2.1it/s 1:50<0.5ss
  Epoch  44/100  |  loss: 12.3027  |  mAP@50: 0.6270  |  mAP@50-95: 0.2541  |  lr: 9.57e-04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 4.6it/s 8.2s0.2s
                   all        600       1200      0.733      0.742      0.687      0.275
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 24, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

44 epochs completed in 1.266 hours.
Optimizer stripped from C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\nextgen\rtdetr_run\detect\weights\last.pt, 66.2MB
Optimizer stripped from C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\code

### A3. Inference — Crop Courtesy & Legal Regions

In [3]:
from ultralytics import RTDETR

BEST_A = NEXTGEN / 'rtdetr_run' / 'detect' / 'weights' / 'best.pt'
if not BEST_A.exists():
    BEST_A = 'rtdetr-l.pt'
    print('WARNING: trained weights not found — using pretrained RT-DETR.')

model_a_inf = RTDETR(str(BEST_A))

CROP_DIR = NEXTGEN / 'crops'
for cls_name in ('courtesy', 'legal'):
    (CROP_DIR / cls_name).mkdir(parents=True, exist_ok=True)


def crop_and_save(img_path: Path, preds, out_dir: Path, stem: str):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    records = []
    for box in preds.boxes:
        cls_id = int(box.cls.item())
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop = img.crop((x1, y1, x2, y2))
        cls_name = 'legal' if cls_id == 0 else 'courtesy'
        save_path = out_dir / cls_name / f'{stem}.png'
        crop.save(save_path)
        records.append({'stem': stem, 'cls': cls_id, 'crop_path': str(save_path)})
    return records


# Also crop training images (needed for Parts B & C fine-tuning)
TRAIN_CROP = NEXTGEN / 'train_crops'
for cls_name in ('courtesy', 'legal'):
    (TRAIN_CROP / cls_name).mkdir(parents=True, exist_ok=True)

all_crop_records = []
for stem in tqdm(test_stems, desc='Cropping test images'):
    img_path = TEST_IMAGES / f'{stem}.tif'
    img_rgb = Image.open(img_path).convert('RGB')  # ensure 3-channel input for RT-DETR
    preds = model_a_inf.predict(img_rgb, conf=0.25, verbose=False)[0]
    all_crop_records.extend(crop_and_save(img_path, preds, CROP_DIR, stem))

# For training data: use ground-truth bounding boxes (faster and more accurate)
for stem in tqdm(train_stems, desc='Cropping train images (GT boxes)'):
    img_path = TRAIN_IMAGES / f'{stem}.tif'
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    for cls_id, cx, cy, bw, bh in parse_bbox(TRAIN_BBOX / f'{stem}.txt'):
        x0 = int((cx - bw / 2) * W)
        y0 = int((cy - bh / 2) * H)
        x1 = int((cx + bw / 2) * W)
        y1 = int((cy + bh / 2) * H)
        crop = img.crop((max(0, x0), max(0, y0), min(W, x1), min(H, y1)))
        cls_name = 'legal' if cls_id == 0 else 'courtesy'
        crop.save(TRAIN_CROP / cls_name / f'{stem}.png')

courtesy_crops = [r for r in all_crop_records if r['cls'] == 1]
legal_crops    = [r for r in all_crop_records if r['cls'] == 0]
print(f'Test crops — Courtesy: {len(courtesy_crops)}  |  Legal: {len(legal_crops)}')

with open(NEXTGEN / 'crop_records.json', 'w') as f:
    json.dump(all_crop_records, f, indent=2)


Cropping test images:   0%|          | 0/600 [00:00<?, ?it/s]

Cropping train images (GT boxes):   0%|          | 0/1799 [00:00<?, ?it/s]

Test crops — Courtesy: 614  |  Legal: 638


### A4. Evaluate RT-DETR — IoU@0.5 / 0.75 / 0.90 / mAP50-95


In [6]:
# ── Run ultralytics validation on the test split ─────────────────────────────
val_results = model_a_inf.val(
    data=str(yaml_path),
    split='val',
    batch=8,
    imgsz=640,
    rect=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=False,
)

# val_results.box.all_ap  — shape (nc, 10)
# 10 IoU thresholds: 0.50, 0.55, 0.60, ..., 0.95
# val_results.box.ap     — shape (nc,)  mean over IoU (= AP@50:95 per class)
# val_results.box.ap50   — shape (nc,)  AP at IoU=0.50 per class
all_ap = np.array(val_results.box.all_ap)     # (nc, 10)
iou_thresholds = [round(0.50 + i * 0.05, 2) for i in range(all_ap.shape[1])]
mean_ap = all_ap.mean(axis=0)                  # (10,) — mAP at each IoU threshold
map_at  = {t: float(mean_ap[i]) for i, t in enumerate(iou_thresholds)}

map50    = float(val_results.box.map50)
map75    = float(val_results.box.map75)
map90    = map_at.get(0.90, float('nan'))
map50_95 = float(val_results.box.map)

# ── Per-class breakdown ───────────────────────────────────────────────────────
class_names = {0: 'legal', 1: 'courtesy'}
per_class = {}
if hasattr(val_results.box, 'ap_class_index') and len(val_results.box.ap_class_index):
    for idx, cls_id in enumerate(val_results.box.ap_class_index):
        cls_name = class_names.get(int(cls_id), str(cls_id))
        ap_row = all_ap[idx]          # (10,)
        per_class[cls_name] = {
            'AP@50':    float(ap_row[0]),
            'AP@75':    float(ap_row[5]),
            'AP@90':    float(ap_row[8]),
            'AP50-95':  float(ap_row.mean()),
        }

# ── Print report ──────────────────────────────────────────────────────────────
print('─' * 54)
print('  RT-DETR-L   Detection Evaluation (Test Split)')
print('─' * 54)
print(f'  mAP @ IoU=0.50        : {map50:.4f}')
print(f'  mAP @ IoU=0.75        : {map75:.4f}')
print(f'  mAP @ IoU=0.90        : {map90:.4f}')
print(f'  mAP @ IoU=0.50:0.95   : {map50_95:.4f}  ← COCO primary metric')
print('─' * 54)
for cls_name, ap in per_class.items():
    print(f'  [{cls_name:<10}]  AP@50={ap["AP@50"]:.4f}  AP@75={ap["AP@75"]:.4f}  '
          f'AP@90={ap["AP@90"]:.4f}  AP50-95={ap["AP50-95"]:.4f}')
print('─' * 54)

# Full IoU sweep
print('\n  Full IoU sweep (mean across classes):')
for t, v in map_at.items():
    bar = '█' * int(v * 40)
    print(f'    IoU={t:.2f}  {v:.4f}  {bar}')

# Save metrics
part_a_metrics = {
    'mAP@0.50':      round(map50,    4),
    'mAP@0.75':      round(map75,    4),
    'mAP@0.90':      round(map90,    4),
    'mAP@0.50:0.95': round(map50_95, 4),
    'per_class':     {k: {m: round(v, 4) for m, v in vals.items()} for k, vals in per_class.items()},
    'full_sweep':    {str(t): round(v, 4) for t, v in map_at.items()},
}
with open(NEXTGEN / 'partA_detection_metrics.json', 'w') as f:
    json.dump(part_a_metrics, f, indent=2)
print(f'\nMetrics saved to {NEXTGEN / "partA_detection_metrics.json"}')


Ultralytics 8.4.14  Python-3.10.19 torch-2.12.0.dev20260219+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
rt-detr-l summary: 310 layers, 31,987,850 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1566.2165.2 MB/s, size: 139.6 KB)
val: Scanning C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\nextgen\rtdetr_data\labels\val.cache... 600 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 600/600  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 75/75 6.4it/s 11.7s0.2s
                   all        600       1200       0.69      0.723      0.626      0.293
Speed: 1.0ms preprocess, 15.2ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\runs\detect\val5
──────────────────────────────────────────────────────
  RT-DETR-L   Detection Evaluation (Test Split)
─────────

---
## Part B — Digit Recognition with TrOCR

TrOCR (`microsoft/trocr-base-handwritten`) combines:
- **ViT encoder** — patch-based image feature extraction
- **Transformer decoder** — autoregressive sequence generation

**Strategy:** Freeze the encoder; fine-tune only the decoder on courtesy-amount crops
to avoid overfitting on the small dataset while leveraging pretrained visual features.

### B1. Dataset & Label Loading

In [9]:
import ast
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel


def load_labels(txt_path: Path) -> dict:
    """Load courtesy-amount labels.

    Keys  : image stems ('ac00000'), matching saved crop filenames.
    Values: digit string ('48626'), compatible with parse_courtesy_int().

    Label file format (tab-separated):
        Cac00000.tif    [10, 4, 8, 6, 2, 6, 10]
    where 10 = <BOS/EOS>, 11 = <SEP> (riyal/halalas), 0-9 = digit.
    """
    labels = {}
    with open(txt_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t', 1)
            if len(parts) < 2:
                continue
            # Normalise key: 'Cac00000.tif' -> 'ac00000'
            stem = Path(parts[0].strip()).stem   # drop .tif
            if stem.startswith('C'):
                stem = stem[1:]                  # drop leading 'C'
            # Decode token list -> label string for TrOCR
            try:
                tokens = ast.literal_eval(parts[1].strip())
            except Exception:
                continue
            chars = []
            for t in tokens:
                if t == 10:
                    chars.append('<BOS/EOS>')
                elif t == 11:
                    chars.append('<SEP>')
                elif isinstance(t, int) and 0 <= t <= 9:
                    chars.append(str(t))
            label_str = ''.join(chars)
            if label_str:
                labels[stem] = label_str
    return labels


train_ca = load_labels(TRAIN_CA)
test_ca  = load_labels(TEST_CA)
print(f'Courtesy labels — train: {len(train_ca)}  |  test: {len(test_ca)}')
print('Sample:', list(test_ca.items())[:3])

# ── Diagnostic: check crop directories ───────────────────────────────────────
for desc, d in [
    ('TRAIN_CROP/courtesy', TRAIN_CROP / 'courtesy'),
    ('CROP_DIR/courtesy',   CROP_DIR   / 'courtesy'),
]:
    pngs = list(d.glob('*.png')) if d.exists() else []
    print(f'  {desc}: {len(pngs)} PNG files  (dir exists: {d.exists()})')
    if pngs:
        print(f'    first 3: {[p.name for p in pngs[:3]]}')

train_crop_stems = {p.stem for p in (TRAIN_CROP / 'courtesy').glob('*.png')} if (TRAIN_CROP / 'courtesy').exists() else set()
test_crop_stems  = {p.stem for p in (CROP_DIR   / 'courtesy').glob('*.png')} if (CROP_DIR   / 'courtesy').exists() else set()
print(f'  Label/crop overlap — train: {len(set(train_ca) & train_crop_stems)}  |  test: {len(set(test_ca) & test_crop_stems)}')


class CourtesyDataset(TorchDataset):
    """TrOCR dataset for handwritten digit courtesy amounts."""

    def __init__(self, crop_dir: Path, labels: dict, processor, max_len: int = 32):
        self.processor = processor
        self.max_len   = max_len
        self.samples   = [
            (str(crop_dir / f'{stem}.png'), lbl)
            for stem, lbl in labels.items()
            if (crop_dir / f'{stem}.png').exists()
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image        = Image.open(path).convert('RGB')
        pixel_values = self.processor(images=image, return_tensors='pt').pixel_values.squeeze(0)
        # as_target_tokenizer() removed — deprecated and removed in transformers>=4.46
        enc = self.processor.tokenizer(
            label, padding='max_length', max_length=self.max_len,
            truncation=True, return_tensors='pt',
        )
        labels_ids = enc.input_ids.squeeze(0)
        labels_ids[labels_ids == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels_ids}


BASE_TROCR = 'microsoft/trocr-base-handwritten'
processor_b = TrOCRProcessor.from_pretrained(BASE_TROCR)

ds_train_b = CourtesyDataset(TRAIN_CROP / 'courtesy', train_ca, processor_b)
ds_val_b   = CourtesyDataset(CROP_DIR   / 'courtesy', test_ca,  processor_b)
print(f'Train B: {len(ds_train_b)}  |  Val B: {len(ds_val_b)}')

if len(ds_train_b) == 0:
    raise RuntimeError(
        'No training crops found. Re-run the A3 cropping cell first, '
        'then re-run this cell.'
    )


Courtesy labels — train: 1738  |  test: 600
Sample: [('ac03000', '<BOS/EOS>10000<BOS/EOS>'), ('ac03001', '<BOS/EOS>5000<BOS/EOS>'), ('ac03003', '<BOS/EOS>5000<BOS/EOS>')]
  TRAIN_CROP/courtesy: 1799 PNG files  (dir exists: True)
    first 3: ['ac00000.png', 'ac00001.png', 'ac00002.png']
  CROP_DIR/courtesy: 600 PNG files  (dir exists: True)
    first 3: ['ac03000.png', 'ac03001.png', 'ac03003.png']
  Label/crop overlap — train: 1724  |  test: 600
Train B: 1724  |  Val B: 600


### B2. Fine-Tune TrOCR for Digits

In [10]:
import evaluate as hf_evaluate
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator,
    TrainerCallback,
)

MODEL_B_DIR = str(NEXTGEN / 'trocr_courtesy')

model_b = VisionEncoderDecoderModel.from_pretrained(BASE_TROCR)
model_b.config.decoder_start_token_id = processor_b.tokenizer.cls_token_id
model_b.config.pad_token_id           = processor_b.tokenizer.pad_token_id
model_b.config.vocab_size             = model_b.config.decoder.vocab_size

# Freeze encoder — fine-tune decoder only to avoid overfitting on small dataset
for param in model_b.encoder.parameters():
    param.requires_grad = False

cer_metric = hf_evaluate.load('cer')


def compute_metrics_b(pred):
    label_ids = pred.label_ids.copy()
    pred_ids  = pred.predictions
    label_ids[label_ids == -100] = processor_b.tokenizer.pad_token_id
    pred_str  = processor_b.batch_decode(pred_ids,   skip_special_tokens=True)
    label_str = processor_b.batch_decode(label_ids,  skip_special_tokens=True)
    return {'cer': cer_metric.compute(predictions=pred_str, references=label_str)}


class EpochProgressCallback(TrainerCallback):
    """Print a one-line summary at the end of every epoch."""

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return
        epoch      = int(state.epoch) if state.epoch else '?'
        total      = args.num_train_epochs
        train_loss = state.log_history[-1].get('loss', float('nan')) if state.log_history else float('nan')
        eval_loss  = metrics.get('eval_loss', float('nan'))
        cer        = metrics.get('eval_cer', float('nan'))
        print(
            f'  Epoch {epoch:>3}/{total}'
            f'  |  train_loss: {train_loss:.4f}'
            f'  |  eval_loss: {eval_loss:.4f}'
            f'  |  CER: {cer:.4f}'
        )


args_b = Seq2SeqTrainingArguments(
    output_dir=MODEL_B_DIR,
    num_train_epochs=30,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    eval_strategy='epoch',        # renamed from evaluation_strategy in transformers>=4.46
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),   # AMP — RTX 5060 Tensor Cores
    dataloader_num_workers=0,         # 0 avoids Windows DataLoader worker deadlocks
    logging_steps=10,
    report_to='none',
    # disable_tqdm removed — keep progress bar so you can see step-level activity
)

trainer_b = Seq2SeqTrainer(
    model=model_b,
    args=args_b,
    train_dataset=ds_train_b,
    eval_dataset=ds_val_b,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics_b,
    callbacks=[EpochProgressCallback()],
)

trainer_b.train()
trainer_b.save_model(MODEL_B_DIR)
processor_b.save_pretrained(MODEL_B_DIR)
print(f'Part B model saved to {MODEL_B_DIR}')


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Cer
1,0.353832,1.345665,0.231872
2,0.252605,1.412906,0.223220
3,0.172761,1.727196,0.226269
4,0.078386,1.884692,0.221078
5,0.054602,1.739950,0.222561
6,0.053178,1.898862,0.217040
7,0.041170,1.802809,0.218276
8,0.021227,1.841241,0.223055
9,0.026621,1.917742,0.216875
10,0.014563,2.221016,0.221819


c:\Users\Admin\miniconda3\envs\warehouse_ai\lib\site-packages\transformers\generation\utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


  Epoch   1/30  |  train_loss: nan  |  eval_loss: 1.3457  |  CER: 0.2319


...

  Epoch  30/30  |  train_loss: nan  |  eval_loss: 2.2627  |  CER: 0.2125


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Part B model saved to C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\nextgen\trocr_courtesy


### B3. Evaluate Part B

In [15]:
import sys; sys.path.insert(0, str(Path(__file__).parent) if '__file__' in dir() else '.')
from utils import courtesy_summary

model_b_best = VisionEncoderDecoderModel.from_pretrained(MODEL_B_DIR).to(DEVICE)
model_b_best.eval()
model_b_best.generation_config.max_length = None  # avoid conflict with max_new_tokens
proc_b_eval  = TrOCRProcessor.from_pretrained(MODEL_B_DIR)

partb_ng = []
for stem in tqdm(test_stems, desc='Part B inference'):
    crop_path = CROP_DIR / 'courtesy' / f'{stem}.png'
    if not crop_path.exists():
        continue
    image        = Image.open(crop_path).convert('RGB')
    pixel_values = proc_b_eval(images=image, return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        generated = model_b_best.generate(pixel_values, max_new_tokens=32)
    predicted = proc_b_eval.batch_decode(generated, skip_special_tokens=True)[0].strip()
    reference = test_ca.get(stem, '')
    partb_ng.append({'stem': stem, 'predicted': predicted, 'reference': reference,
                     'file': f'{stem}.png'})

with open(NEXTGEN / 'partB_nextgen_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(partb_ng, f, indent=2, ensure_ascii=False)

preds_b = [r['predicted'] for r in partb_ng if r['reference']]
refs_b  = [r['reference'] for r in partb_ng if r['reference']]

metrics_b = courtesy_summary(refs_b, preds_b)
print(f'Part B (TrOCR courtesy amounts)  —  {len(preds_b)} samples')
print(f'  Digit accuracy  : {metrics_b["digit_accuracy"]}%')
print(f'  % no errors     : {metrics_b["pct_no_error"]}%')
print(f'  % one error     : {metrics_b["pct_one_error"]}%')
print(f'  % two+ errors   : {metrics_b["pct_two_plus"]}%')


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Part B inference:   0%|          | 0/600 [00:00<?, ?it/s]

Part B (TrOCR courtesy amounts)  —  600 samples
  Digit accuracy  : 78.82%
  % no errors     : 32.33%
  % one error     : 22.33%
  % two+ errors   : 45.33%


---
## Part C — Arabic Legal Text with TrOCR

### Mandatory pre-processing (paper recipe)

| Step | Why |
|---|---|
| **Horizontal flip** | Arabic is RTL; Transformer decoder expects LTR sequences |
| **4× augmentation** | Elastic transform + rotation ±15° + zoom — matches paper’s 4× expansion |
| **Char-level tokeniser** | 24 Arabic chars + space; replaces BPE sub-words; reduces vocabulary complexity |
| **Spelling normalisation** | Unify variant spellings (\u0645\u0627\u0626\u0647→\u0645\u0627\u0626\u0629, \u0627\u0644\u0641→\u0623\u0644\u0641) before training |

### C1. Spelling Normalisation

In [19]:
SPELLING_MAP = {
    '\u0645\u0627\u0626\u0647':   '\u0645\u0627\u0626\u0629',    # \u0645\u0627\u0626\u0647 → \u0645\u0627\u0626\u0629
    '\u0645\u0626\u0647':    '\u0645\u0627\u0626\u0629',    # \u0645\u0626\u0647 → \u0645\u0627\u0626\u0629
    '\u0645\u0626\u0629':    '\u0645\u0627\u0626\u0629',    # \u0645\u0626\u0629 → \u0645\u0627\u0626\u0629
    '\u0627\u0644\u0641':    '\u0623\u0644\u0641',     # \u0627\u0644\u0641 → \u0623\u0644\u0641
    '\u0622\u0644\u0641':    '\u0623\u0644\u0641',     # \u0622\u0644\u0641 → \u0623\u0644\u0641
    '\u0623\u0644\u0627\u0641':   '\u0622\u0644\u0627\u0641',    # \u0623\u0644\u0627\u0641 → \u0622\u0644\u0627\u0641
    '\u0627\u062b\u0646\u064a\u0646':  '\u0627\u062b\u0646\u0627\u0646',   # \u0627\u062b\u0646\u064a\u0646 → \u0627\u062b\u0646\u0627\u0646
    '\u062b\u0644\u0627\u062b\u0647':  '\u062b\u0644\u0627\u062b\u0629',   # \u062b\u0644\u0627\u062b\u0647 → \u062b\u0644\u0627\u062b\u0629
    '\u0627\u0631\u0628\u0639\u0647':  '\u0623\u0631\u0628\u0639\u0629',   # \u0627\u0631\u0628\u0639\u0647 → \u0623\u0631\u0628\u0639\u0629
    '\u0627\u0631\u0628\u0639':   '\u0623\u0631\u0628\u0639',    # \u0627\u0631\u0628\u0639 → \u0623\u0631\u0628\u0639
    '\u062e\u0645\u0633\u0647':   '\u062e\u0645\u0633\u0629',    # \u062e\u0645\u0633\u0647 → \u062e\u0645\u0633\u0629
    '\u0633\u062a\u0647':    '\u0633\u062a\u0629',     # \u0633\u062a\u0647 → \u0633\u062a\u0629
    '\u0633\u0628\u0639\u0647':   '\u0633\u0628\u0639\u0629',    # \u0633\u0628\u0639\u0647 → \u0633\u0628\u0639\u0629
    '\u062b\u0645\u0627\u0646\u064a\u0647': '\u062b\u0645\u0627\u0646\u064a\u0629',  # \u062b\u0645\u0627\u0646\u064a\u0647 → \u062b\u0645\u0627\u0646\u064a\u0629
    '\u062a\u0633\u0639\u0647':   '\u062a\u0633\u0639\u0629',    # \u062a\u0633\u0639\u0647 → \u062a\u0633\u0639\u0629
    '\u0639\u0634\u0631\u0647':   '\u0639\u0634\u0631\u0629',    # \u0639\u0634\u0631\u0647 → \u0639\u0634\u0631\u0629
    '\u0631\u064a\u0627\u0644\u0627':   '\u0631\u064a\u0627\u0644',     # \u0631\u064a\u0627\u0644\u0627 → \u0631\u064a\u0627\u0644
    '\u0631\u064a\u0627\u0644\u0647':   '\u0631\u064a\u0627\u0644',     # \u0631\u064a\u0627\u0644\u0647 → \u0631\u064a\u0627\u0644
    '\u0647\u0644\u0644\u0647':   '\u0647\u0644\u0644\u0629',    # \u0647\u0644\u0644\u0647 → \u0647\u0644\u0644\u0629
}


def normalize_arabic(text: str) -> str:
    for wrong, correct in SPELLING_MAP.items():
        text = text.replace(wrong, correct)
    # Normalise alef variants to plain alef
    text = re.sub('[\u0625\u0623\u0622\u0627]', '\u0627', text)
    # Remove tatweel (elongation)
    text = text.replace('\u0640', '')
    # Strip diacritics (harakat)
    text = re.sub('[\u064b-\u065f]', '', text)
    return text.strip()


# Quick sanity check
samples = ['\u0645\u0627\u0626\u0647 \u0648\u062e\u0645\u0633\u0647 \u0648\u0639\u0634\u0631\u0648\u0646', '\u0627\u0644\u0641 \u0648\u062b\u0644\u0627\u062b\u0647 \u0645\u0627\u0626\u0629']
for s in samples:
    print(f'  {s!r}')
    print(f'    -> {normalize_arabic(s)!r}')

  'مائه وخمسه وعشرون'
    -> 'مائة وخمسة وعشرون'
  'الف وثلاثه مائة'
    -> 'الف وثلاثة مائة'


### C2. Character-Level Tokeniser (24 Arabic Labels)

In [20]:
import ast, re as _re

# Load all legal labels to derive the actual character set used in the data
def load_la_labels(la_path: Path) -> dict:
    labels = {}
    with open(la_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) < 2:
                continue
            # Normalize key: 'Lac00000.tif' → 'ac00000' (strip leading 'L' + '.tif')
            stem = Path(parts[0].strip()).stem
            if stem.startswith('L'):
                stem = stem[1:]
            # Value is a Python list string with RTL/LTR control chars: strip then eval
            raw = parts[1].strip()
            raw = _re.sub(r'[\u200e\u200f\u202a-\u202e\u2066-\u2069]', '', raw)
            try:
                tokens = ast.literal_eval(raw)
                label  = normalize_arabic(' '.join(tokens))
            except Exception:
                continue
            if label:
                labels[stem] = label
    return labels


train_la = load_la_labels(TRAIN_LA)
test_la  = load_la_labels(TEST_LA)
print(f'Legal labels — train: {len(train_la)}  |  test: {len(test_la)}')

# Build character vocabulary from the actual training labels
all_chars = set()
for lbl in train_la.values():
    all_chars.update(lbl)
ARABIC_CHARS = sorted(all_chars - set('\n\r\t'))

SPECIAL = ['<pad>', '<sos>', '<eos>', '<unk>']
VOCAB   = SPECIAL + ARABIC_CHARS
CHAR2ID = {c: i for i, c in enumerate(VOCAB)}
ID2CHAR = {i: c for c, i in CHAR2ID.items()}
VOCAB_SIZE = len(VOCAB)

PAD_ID = CHAR2ID['<pad>']
SOS_ID = CHAR2ID['<sos>']
EOS_ID = CHAR2ID['<eos>']
UNK_ID = CHAR2ID['<unk>']

print(f'Vocabulary size : {VOCAB_SIZE}  (including 4 special tokens)')
print(f'Characters      : {ARABIC_CHARS}')


def encode_label(text: str, max_len: int = 64) -> List[int]:
    ids = [SOS_ID] + [CHAR2ID.get(c, UNK_ID) for c in text] + [EOS_ID]
    ids = ids[:max_len]
    ids += [PAD_ID] * (max_len - len(ids))
    return ids


def decode_ids(ids) -> str:
    chars = []
    for i in ids:
        c = ID2CHAR.get(int(i), '')
        if c in ('<eos>', '<pad>'):
            break
        if c not in ('<sos>', '<unk>'):
            chars.append(c)
    return ''.join(chars)


Legal labels — train: 1799  |  test: 600
Vocabulary size : 53  (including 4 special tokens)
Characters      : [' ', '#', '/', '0', '1', '4', '6', '،', 'ء', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ر', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي', '٠', '١', '٢', '٣', '٤', '٥', '٦', '٧', '٨', '٩', '⁄']


### C3. Dataset with 4× Augmentation + Horizontal Flip

In [21]:
try:
    import albumentations as A
    HAS_ALB = True
except ImportError:
    HAS_ALB = False
    print('albumentations not found — augmentation disabled.')

# 3 augmentation transforms; combined with the original => 4× dataset
_AUG_TRANSFORMS = [
    A.Compose([A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1.0,
                                   border_mode=0)]),
    A.Compose([A.Rotate(limit=15, p=1.0, border_mode=0)]),
    A.Compose([A.RandomScale(scale_limit=0.2, p=1.0)]),
] if HAS_ALB else []


def augment_pil(image: Image.Image) -> List[Image.Image]:
    """Return the original + up to 3 augmented PIL variants."""
    variants = [image]
    arr = np.array(image)
    for t in _AUG_TRANSFORMS:
        aug = t(image=arr)['image']
        variants.append(Image.fromarray(aug))
    return variants  # 4 images


class ArabicLegalDataset(TorchDataset):
    """TrOCR dataset for handwritten Arabic legal amounts.

    Always applies horizontal flip (RTL -> LTR) before encoding.
    If augment=True, expands each sample to 4 variants.
    """

    def __init__(self, crop_dir: Path, labels: dict, processor,
                 augment: bool = False, max_len: int = 64):
        self.processor = processor
        self.max_len   = max_len
        self.samples: List[Tuple[Image.Image, str]] = []

        for stem, label in labels.items():
            path = crop_dir / f'{stem}.png'
            if not path.exists():
                continue
            img = ImageOps.mirror(Image.open(path).convert('RGB'))  # H-flip: RTL->LTR
            if augment:
                for variant in augment_pil(img):
                    self.samples.append((variant, label))
            else:
                self.samples.append((img, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image, label = self.samples[idx]
        pixel_values = self.processor(images=image, return_tensors='pt').pixel_values.squeeze(0)
        label_ids    = torch.tensor(encode_label(label, self.max_len), dtype=torch.long)
        masked       = label_ids.clone()
        masked[masked == PAD_ID] = -100
        return {'pixel_values': pixel_values, 'labels': masked}


processor_c = TrOCRProcessor.from_pretrained(BASE_TROCR)

ds_train_c = ArabicLegalDataset(TRAIN_CROP / 'legal', train_la, processor_c, augment=True)
ds_val_c   = ArabicLegalDataset(CROP_DIR   / 'legal', test_la,  processor_c, augment=False)
print(f'Train C (with 4x aug): {len(ds_train_c)}  |  Val C: {len(ds_val_c)}')

C:\Users\Admin\AppData\Local\Temp\ipykernel_9192\1685880511.py:10: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.Compose([A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1.0,


Train C (with 4x aug): 7192  |  Val C: 600


### C4. Fine-Tune TrOCR for Arabic

In [24]:
MODEL_C_DIR = str(NEXTGEN / 'trocr_legal')

model_c = VisionEncoderDecoderModel.from_pretrained(BASE_TROCR)

# Replace the decoder token embeddings to match our char vocabulary
model_c.decoder.resize_token_embeddings(VOCAB_SIZE)
model_c.config.decoder_start_token_id = SOS_ID
model_c.config.pad_token_id           = PAD_ID
model_c.config.eos_token_id           = EOS_ID
model_c.config.vocab_size             = VOCAB_SIZE

# Freeze encoder — fine-tune decoder only
for param in model_c.encoder.parameters():
    param.requires_grad = False

# Pre-load once — avoids re-loading from disk on every eval epoch
cer_metric_c = hf_evaluate.load('cer')


def compute_metrics_c(pred):
    label_ids = pred.label_ids.copy()
    pred_ids  = pred.predictions
    label_ids[label_ids == -100] = PAD_ID
    pred_str  = [decode_ids(p) for p in pred_ids]
    label_str = [decode_ids(l) for l in label_ids]
    return {'cer': cer_metric_c.compute(predictions=pred_str, references=label_str)}


class EpochProgressCallbackC(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return
        epoch      = int(state.epoch) if state.epoch else '?'
        total      = args.num_train_epochs
        train_loss = state.log_history[-1].get('loss', float('nan')) if state.log_history else float('nan')
        eval_loss  = metrics.get('eval_loss', float('nan'))
        cer        = metrics.get('eval_cer', float('nan'))
        print(
            f'  Epoch {epoch:>3}/{total}'
            f'  |  train_loss: {train_loss:.4f}'
            f'  |  eval_loss: {eval_loss:.4f}'
            f'  |  CER: {cer:.4f}'
        )


args_c = Seq2SeqTrainingArguments(
    output_dir=MODEL_C_DIR,
    num_train_epochs=30,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    generation_max_length=64,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    # gradient_checkpointing removed — incompatible with VisionEncoderDecoderModel
    # in newer PyTorch (CheckpointError: different tensor count on recompute).
    # Not needed here since the encoder is frozen — only the decoder trains.
    dataloader_num_workers=0,     # avoid Windows DataLoader deadlocks
    logging_steps=20,
    report_to='none',
)

trainer_c = Seq2SeqTrainer(
    model=model_c,
    args=args_c,
    train_dataset=ds_train_c,
    eval_dataset=ds_val_c,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics_c,
    callbacks=[EpochProgressCallbackC()],
)

trainer_c.train()
trainer_c.save_model(MODEL_C_DIR)
processor_c.save_pretrained(MODEL_C_DIR)
print(f'Part C model saved to {MODEL_C_DIR}')


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### C5. Evaluate Part C

In [ ]:
model_c_best = VisionEncoderDecoderModel.from_pretrained(MODEL_C_DIR).to(DEVICE)
model_c_best.eval()
proc_c_eval  = TrOCRProcessor.from_pretrained(MODEL_C_DIR)

partc_ng = []
for stem in tqdm(test_stems, desc='Part C inference'):
    crop_path = CROP_DIR / 'legal' / f'{stem}.png'
    if not crop_path.exists():
        continue
    image = ImageOps.mirror(Image.open(crop_path).convert('RGB'))  # RTL -> LTR
    pixel_values = proc_c_eval(images=image, return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        generated = model_c_best.generate(
            pixel_values,
            max_new_tokens=64,
            decoder_start_token_id=SOS_ID,
        )
    predicted = decode_ids(generated[0].cpu().tolist())
    reference = test_la.get(stem, '')
    partc_ng.append({'stem': stem, 'predicted': predicted, 'reference': reference,
                     'file': f'{stem}.png'})

with open(NEXTGEN / 'partC_nextgen_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(partc_ng, f, indent=2, ensure_ascii=False)

preds_c = [r['predicted'] for r in partc_ng if r['reference']]
refs_c  = [r['reference'] for r in partc_ng if r['reference']]
cer_c   = jiwer.cer(refs_c, preds_c)
wer_c   = jiwer.wer(refs_c, preds_c)
print(f'Part C (TrOCR Arabic)  —  CER: {cer_c:.4f}  |  WER: {wer_c:.4f}')
print(f'Evaluated on {len(preds_c)} samples')

---
## Part D — Mutual Enhancement: Levenshtein + Magnitude Correction

### Algorithm

1. **Fuzzy Arabic word matching** — each predicted token is matched against the monetary
   vocabulary using Levenshtein distance (threshold ≤ 2). Corrects OCR character substitutions.
2. **Legal → integer** — corrected tokens are passed to `legal_text_to_digits`.
3. **Magnitude correction** — if a magnitude keyword (`\u0623\u0644\u0641`, `\u0645\u0627\u0626\u0629`, `\u0645\u0644\u064a\u0648\u0646`) is detected in the legal text,
   verify the courtesy integer is consistent; flag potential *missing-zero* errors.
4. **Cross-verification** — cascaded fallback: HIGH / MEDIUM / COURTESY\_ONLY /
   LEGAL\_ONLY / LOW / UNPARSEABLE.

### D1. Levenshtein Distance

In [ ]:
def levenshtein_distance(s1: str, s2: str) -> int:
    """Efficient character-level edit distance."""
    if s1 == s2:
        return 0
    m, n = len(s1), len(s2)
    if m == 0:
        return n
    if n == 0:
        return m
    row = list(range(n + 1))
    for i in range(1, m + 1):
        prev, row[0] = row[0], i
        for j in range(1, n + 1):
            temp = row[j]
            if s1[i - 1] == s2[j - 1]:
                row[j] = prev
            else:
                row[j] = 1 + min(prev, row[j], row[j - 1])
            prev = temp
    return row[n]


# Arabic monetary vocabulary (canonical forms)
MONETARY_VOCAB = [
    '\u0635\u0641\u0631', '\u0648\u0627\u062d\u062f', '\u0648\u0627\u062d\u062f\u0629',
    '\u0627\u062b\u0646\u0627\u0646', '\u0627\u062b\u0646\u062a\u0627\u0646',
    '\u062b\u0644\u0627\u062b\u0629', '\u0623\u0631\u0628\u0639\u0629', '\u062e\u0645\u0633\u0629',
    '\u0633\u062a\u0629', '\u0633\u0628\u0639\u0629', '\u062b\u0645\u0627\u0646\u064a\u0629',
    '\u062a\u0633\u0639\u0629', '\u0639\u0634\u0631\u0629', '\u0639\u0634\u0631\u0648\u0646',
    '\u062b\u0644\u0627\u062b\u0648\u0646', '\u0623\u0631\u0628\u0639\u0648\u0646',
    '\u062e\u0645\u0633\u0648\u0646', '\u0633\u062a\u0648\u0646', '\u0633\u0628\u0639\u0648\u0646',
    '\u062b\u0645\u0627\u0646\u0648\u0646', '\u062a\u0633\u0639\u0648\u0646',
    '\u0645\u0627\u0626\u0629', '\u0645\u0627\u0626\u062a\u0627\u0646',
    '\u0623\u0644\u0641', '\u0623\u0644\u0641\u0627\u0646', '\u0622\u0644\u0627\u0641',
    '\u0645\u0644\u064a\u0648\u0646', '\u0631\u064a\u0627\u0644', '\u0647\u0644\u0644\u0629', '\u0648',
]

MAGNITUDE_KEYWORDS = {
    '\u0645\u0644\u064a\u0648\u0646':  1_000_000,
    '\u0622\u0644\u0627\u0641':   1_000,
    '\u0623\u0644\u0641\u0627\u0646':  2_000,
    '\u0623\u0644\u0641':    1_000,
    '\u0645\u0627\u0626\u062a\u0627\u0646': 200,
    '\u0645\u0627\u0626\u0629':   100,
}


def fuzzy_correct_tokens(tokens: List[str], threshold: int = 2) -> List[str]:
    """Replace each token with the closest monetary-vocabulary word (if dist <= threshold)."""
    corrected = []
    for tok in tokens:
        if tok in MONETARY_VOCAB:
            corrected.append(tok)
            continue
        best, best_d = tok, threshold + 1
        for v in MONETARY_VOCAB:
            d = levenshtein_distance(tok, v)
            if d < best_d:
                best_d, best = d, v
        corrected.append(best if best_d <= threshold else tok)
    return corrected


# Quick sanity check
test_toks = ['\u0645\u0627\u0626\u0647', '\u062e\u0645\u0633\u0647', '\u0627\u0644\u0627\u0641', '\u0631\u064a\u0627\u0644\u0627']
corrected  = fuzzy_correct_tokens(test_toks)
print('Fuzzy correction:')
for orig, corr in zip(test_toks, corrected):
    mark = '\u2713' if orig != corr else '='
    print(f'  {mark}  {orig!r}  ->  {corr!r}')

### D2. Magnitude Correction

In [ ]:
def parse_courtesy_int(predicted_str: str) -> Optional[int]:
    s = predicted_str.replace('<BOS/EOS>', '')
    for sep in ('<SEP>', '.'):
        if sep in s:
            s = s.split(sep)[0]
    digits = ''.join(c for c in s if c.isdigit())
    if not digits:
        return None
    val = int(digits)
    return val if val > 0 else None


def magnitude_correct(
    courtesy_int: Optional[int],
    legal_tokens: List[str],
) -> Tuple[Optional[int], bool]:
    """
    Use the highest magnitude keyword in legal_tokens to detect missing-zero errors
    in the courtesy amount (e.g., 500 predicted when legal says \u0623\u0644\u0641 -> should be 5000).

    Returns (corrected_amount, was_corrected).
    """
    if courtesy_int is None:
        return None, False

    corrected_tokens = fuzzy_correct_tokens(legal_tokens)
    max_mag = None
    for tok in corrected_tokens:
        if tok in MAGNITUDE_KEYWORDS:
            mag = MAGNITUDE_KEYWORDS[tok]
            if max_mag is None or mag > max_mag:
                max_mag = mag

    if max_mag is None:
        return courtesy_int, False

    # If the courtesy amount is less than the magnitude by a factor of ~10,
    # it likely has a missing trailing zero.
    if courtesy_int * 10 >= max_mag and courtesy_int < max_mag:
        return courtesy_int * 10, True

    return courtesy_int, False


# Sanity check
print('Magnitude correction test:')
test_cases = [
    (500,  ['\u0623\u0644\u0641', '\u0648', '\u062e\u0645\u0633\u0629', '\u0645\u0627\u0626\u0629']),   # Should detect: 500 * 10 = 5000 >= 1000
    (1500, ['\u0623\u0644\u0641', '\u0648', '\u062e\u0645\u0633\u0629', '\u0645\u0627\u0626\u0629']),   # 1500 >= 1000, no correction needed
    (200,  ['\u0645\u0627\u0626\u0629']),                              # 200 * 10 = 2000 >= 100 but 200 >= 100 too
]
for amt, toks in test_cases:
    result, corrected = magnitude_correct(amt, toks)
    print(f'  courtesy={amt}, legal={toks[:2]}...  =>  {result}  (corrected={corrected})')

### D3. Full Cross-Verification Pipeline

In [ ]:
# Load next-gen predictions
with open(NEXTGEN / 'partB_nextgen_predictions.json', encoding='utf-8') as f:
    partb_ng = json.load(f)
with open(NEXTGEN / 'partC_nextgen_predictions.json', encoding='utf-8') as f:
    partc_ng = json.load(f)

partb_by_stem = {r['stem']: r for r in partb_ng}
partc_by_stem = {r['stem']: r for r in partc_ng}

results_ng = []

for stem, b_rec in partb_by_stem.items():
    c_rec = partc_by_stem.get(stem)

    # ── Parse courtesy (Part B) ───────────────────────────────────────────────
    courtesy_int     = parse_courtesy_int(b_rec['predicted'])
    courtesy_ref_int = parse_courtesy_int(b_rec['reference'])

    # ── Parse legal (Part C) with fuzzy correction ────────────────────────────
    legal_int     = None
    legal_ref_int = None
    was_corrected = False

    if c_rec is not None:
        raw_tokens   = c_rec['predicted'].split()
        clean_tokens = fuzzy_correct_tokens(raw_tokens)
        legal_int    = legal_text_to_digits(clean_tokens)

        ref_tokens    = c_rec['reference'].split()
        legal_ref_int = legal_text_to_digits(fuzzy_correct_tokens(ref_tokens))

        # Apply magnitude correction to the courtesy amount
        courtesy_int, was_corrected = magnitude_correct(courtesy_int, clean_tokens)

    # ── Verification verdict ──────────────────────────────────────────────────
    if c_rec is not None:
        if courtesy_int is None or legal_int is None:
            verdict = 'UNPARSEABLE'
        elif courtesy_int == legal_int:
            verdict = 'VERIFIED'
        else:
            verdict = 'FAILED'
    else:
        if courtesy_int is None or courtesy_ref_int is None:
            verdict = 'UNPARSEABLE'
        elif courtesy_int == courtesy_ref_int:
            verdict = 'VERIFIED'
        else:
            verdict = 'FAILED'

    # ── Ground-truth verdict ──────────────────────────────────────────────────
    if courtesy_ref_int is not None and legal_ref_int is not None:
        gt_verdict = 'VERIFIED' if courtesy_ref_int == legal_ref_int else 'FAILED'
    else:
        gt_verdict = 'UNPARSEABLE'

    # ── Mutual improvement confidence band ────────────────────────────────────
    if courtesy_int is None and legal_int is None:
        confidence, final_amt = 'UNPARSEABLE', None
    elif courtesy_int is None:
        confidence, final_amt = 'LEGAL_ONLY', legal_int
    elif legal_int is None:
        confidence, final_amt = 'COURTESY_ONLY', courtesy_int
    elif courtesy_int == legal_int:
        confidence, final_amt = 'HIGH', courtesy_int
    else:
        rel_diff = abs(courtesy_int - legal_int) / max(courtesy_int, 1)
        if rel_diff <= 0.10:
            confidence, final_amt = 'MEDIUM', courtesy_int
        else:
            confidence, final_amt = 'LOW', courtesy_int

    ref     = courtesy_ref_int
    correct = (final_amt == ref) if (final_amt is not None and ref is not None) else None

    results_ng.append({
        'stem':             stem,
        'courtesy_int':     courtesy_int,
        'legal_int':        legal_int,
        'courtesy_ref_int': courtesy_ref_int,
        'legal_ref_int':    legal_ref_int,
        'verdict':          verdict,
        'gt_verdict':       gt_verdict,
        'confidence':       confidence,
        'final_amount':     final_amt,
        'correct':          correct,
        'was_corrected':    was_corrected,
    })

total_ng       = len(results_ng)
verified_ng    = sum(1 for r in results_ng if r['verdict'] == 'VERIFIED')
failed_ng      = sum(1 for r in results_ng if r['verdict'] == 'FAILED')
unparseable_ng = sum(1 for r in results_ng if r['verdict'] == 'UNPARSEABLE')
corrected_ng   = sum(1 for r in results_ng if r['was_corrected'])
evaluable_ng   = [r for r in results_ng if r['correct'] is not None]
overall_ng     = sum(1 for r in evaluable_ng if r['correct'])

ng_parseable = [r for r in results_ng if r['verdict'] != 'UNPARSEABLE']
ng_ver_acc   = (sum(1 for r in ng_parseable if r['verdict'] == 'VERIFIED')
                / len(ng_parseable) * 100) if ng_parseable else 0.0
ng_overall_acc = overall_ng / len(evaluable_ng) * 100 if evaluable_ng else 0.0

print('\u2500\u2500 Next-Gen Part D Results \u2500\u2500')
print(f'  Total              : {total_ng}')
print(f'  VERIFIED           : {verified_ng:>4}  ({verified_ng / total_ng * 100:.1f}%)')
print(f'  FAILED             : {failed_ng:>4}  ({failed_ng / total_ng * 100:.1f}%)')
print(f'  UNPARSEABLE        : {unparseable_ng:>4}  ({unparseable_ng / total_ng * 100:.1f}%)')
print(f'  Magnitude corrected: {corrected_ng}')
print(f'  Verification acc.  : {ng_ver_acc:.1f}%  ({sum(1 for r in ng_parseable if r["verdict"]=="VERIFIED")} / {len(ng_parseable)})')
print(f'  Overall accuracy   : {overall_ng} / {len(evaluable_ng)} = {ng_overall_acc:.1f}%')

### D4. Comparison with Baseline

In [ ]:
import numpy as np

baseline = {}
baseline_path = ARTIFACTS / 'partD_metrics.json'
if baseline_path.exists():
    with open(baseline_path) as f:
        baseline = json.load(f)

b_ver_acc = baseline.get('verification_accuracy_pct', float('nan'))
b_overall = baseline.get('overall_accuracy_after_mutual_improvement_pct', float('nan'))

print('\u2500\u2500 Baseline vs. Next-Gen Comparison \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'{"Feature":<42} {"Baseline":>12} {"Next-Gen":>12}')
print('-' * 68)

rows = [
    ('Detection model',              'YOLOv8s',      'RT-DETR-L'),
    ('Digit recognition model',      'CRNN + CTC',   'TrOCR'),
    ('Arabic recognition model',     'CRNN + CTC',   'TrOCR + Arabic'),
    ('Horizontal flip (RTL->LTR)',   'No',           'Yes'),
    ('4x data augmentation',         'No',           'Yes'),
    ('Char-level tokeniser',         'No',           'Yes (24 labels)'),
    ('Spelling normalisation',       'No',           'Yes'),
    ('Levenshtein fuzzy matching',   'No',           'Yes (thr=2)'),
    ('Magnitude correction',         'No',           'Yes'),
    (f'Verification accuracy (%)',   f'{b_ver_acc:.1f}', f'{ng_ver_acc:.1f}'),
    (f'Overall accuracy (%)',        f'{b_overall:.1f}', f'{ng_overall_acc:.1f}'),
]
for label, base_val, ng_val in rows:
    print(f'{label:<42} {base_val:>12} {ng_val:>12}')

# ── Bar chart ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
metrics       = ['Verification Accuracy', 'Overall Accuracy']
baseline_vals = [b_ver_acc, b_overall]
nextgen_vals  = [ng_ver_acc, ng_overall_acc]

x = np.arange(len(metrics))
w = 0.35
bars1 = ax.bar(x - w / 2, baseline_vals, w, label='Baseline (YOLO + CRNN)',      color='#3498db')
bars2 = ax.bar(x + w / 2, nextgen_vals,  w, label='Next-Gen (RT-DETR + TrOCR)', color='#2ecc71')

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 115)
ax.set_title('Baseline vs. Next-Gen Pipeline')
ax.legend()
plt.tight_layout()
plt.savefig(NEXTGEN / 'comparison.png', bbox_inches='tight')
plt.show()

### D5. Save Results

In [ ]:
def make_serialisable(obj):
    if isinstance(obj, dict):
        return {k: make_serialisable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_serialisable(v) for v in obj]
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, bool):
        return obj
    return obj


with open(NEXTGEN / 'partD_nextgen_results.json', 'w', encoding='utf-8') as f:
    json.dump(make_serialisable(results_ng), f, indent=2, ensure_ascii=False)

metrics_out = {
    'model_parts': {
        'A': 'RT-DETR-L',
        'B': 'TrOCR (trocr-base-handwritten, decoder fine-tuned)',
        'C': 'TrOCR + H-flip + 4x aug + char tokeniser + spelling norm',
        'D': 'Levenshtein fuzzy matching + magnitude correction',
    },
    'total_samples':          total_ng,
    'verified':               verified_ng,
    'failed':                 failed_ng,
    'unparseable':            unparseable_ng,
    'magnitude_corrections':  corrected_ng,
    'verification_accuracy_pct':         round(ng_ver_acc, 2),
    'overall_accuracy_pct':              round(ng_overall_acc, 2),
    'comparison': {
        'baseline_ver_acc':  b_ver_acc,
        'nextgen_ver_acc':   round(ng_ver_acc, 2),
        'baseline_overall':  b_overall,
        'nextgen_overall':   round(ng_overall_acc, 2),
    },
}

with open(NEXTGEN / 'partD_nextgen_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)

print('Saved to artifacts/nextgen/')
for fname in [
    'partB_nextgen_predictions.json',
    'partC_nextgen_predictions.json',
    'partD_nextgen_results.json',
    'partD_nextgen_metrics.json',
    'comparison.png',
]:
    size = (NEXTGEN / fname).stat().st_size if (NEXTGEN / fname).exists() else 0
    print(f'  {fname:<45} ({size:,} bytes)')

---
## Summary

### What this notebook improves over the baseline

| Component | Baseline | This Notebook |
|---|---|---|
| **Detection** | YOLOv8s | **RT-DETR-L** — GIoU loss, rect inference, perspective + mosaic aug |
| **Digit OCR** | CRNN + CTC | **TrOCR** — ViT encoder + Transformer decoder |
| **Arabic OCR** | CRNN + CTC | **TrOCR** + H-flip + 4× aug + 24-label char tokeniser + spelling norm |
| **Verification** | Exact match + fallback | **Levenshtein** fuzzy vocab match + **magnitude correction** |

### RTX 5060 Optimisations

| Setting | Effect |
|---|---|
| `amp=True` in RT-DETR | Tensor Cores, ~2× throughput |
| `fp16=True` in TrOCR | Halves VRAM usage |
| `gradient_checkpointing=True` | Trades compute for VRAM in Part C |
| Encoder frozen during fine-tuning | Reduces trainable params by ~60%, cuts VRAM ~40% |

### Expected Gains (vs. paper benchmarks)

- **RT-DETR** tighter boxes → cleaner crops fed to OCR → lower CER/WER.
- **TrOCR attention** handles variable-width Arabic cursive better than CTC-beam-search.
- **Horizontal flip + char tokeniser** directly addresses the two biggest failure modes
  identified in the Part C limitations analysis (resolution + LTR assumption).
- **Magnitude correction** eliminates the most common factor-of-10 errors from missing zeros.
- **Levenshtein fuzzy matching** corrects single-character OCR substitutions in Arabic words
  before they reach the legal-to-integer parser, improving legal-amount accuracy.